In [1]:
import torch.optim as optim
import matplotlib.pyplot as plt

from src.load_and_save import save_model
from src.model import SimpleNN
from src.pruning import get_intermediate_outputs_as_numpy
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.nn as nn
import torch

from src.pruning import is_all_layers_separated
import numpy as np



In [2]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=2048, shuffle=False)

In [3]:
from src.improved_model import SimpleCNN

# Instantiate the model
model = SimpleCNN(0.0).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=float(1e-2))

In [4]:
from src.training import get_average_separation

# Training loop
num_epochs: int = 500
target_accuracy: float = .96
maximum_scramble_distance: float = 5.0

test_data, _ = next(iter(test_dataloader))
test_data.to(device)


for epoch in range(num_epochs):
    for inputs, labels in train_dataloader:
        if epoch == 0:
            break
        # Forward pass
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # separation: float = get_average_separation(test_data, model)

    # Assess progress:
    # data: np.ndarray = get_intermediate_outputs_as_numpy(model, train_dataloader)
    validation_accuracy: float = get_accuracy(model, val_dataloader)
    if validation_accuracy > target_accuracy:
        model.scramble_distance = min(model.scramble_distance + .25, maximum_scramble_distance)
        print(model.scramble_distance)
    print(f'Epoch [{epoch+1}/{num_epochs}], Accuracy: {validation_accuracy:.4f}')


Epoch [1/500], Accuracy: 0.1056
Epoch [2/500], Accuracy: 0.8914
Epoch [3/500], Accuracy: 0.9233
Epoch [4/500], Accuracy: 0.9324
Epoch [5/500], Accuracy: 0.9411
Epoch [6/500], Accuracy: 0.9456
Epoch [7/500], Accuracy: 0.9500
Epoch [8/500], Accuracy: 0.9542
Epoch [9/500], Accuracy: 0.9556
Epoch [10/500], Accuracy: 0.9578
0.25
Epoch [11/500], Accuracy: 0.9607
Epoch [12/500], Accuracy: 0.9594
0.5
Epoch [13/500], Accuracy: 0.9618
0.75
Epoch [14/500], Accuracy: 0.9604
Epoch [15/500], Accuracy: 0.9572
Epoch [16/500], Accuracy: 0.9567
Epoch [17/500], Accuracy: 0.9589
Epoch [18/500], Accuracy: 0.9582
Epoch [19/500], Accuracy: 0.9594
Epoch [20/500], Accuracy: 0.9588
Epoch [21/500], Accuracy: 0.9592
Epoch [22/500], Accuracy: 0.9593
1.0
Epoch [23/500], Accuracy: 0.9619
Epoch [24/500], Accuracy: 0.9574
1.25
Epoch [25/500], Accuracy: 0.9610
Epoch [26/500], Accuracy: 0.9565
Epoch [27/500], Accuracy: 0.9561
Epoch [28/500], Accuracy: 0.9584
Epoch [29/500], Accuracy: 0.9595
Epoch [30/500], Accuracy: 0.9

KeyboardInterrupt: 

In [5]:
def get_signed_accuracy(model: SimpleNN, dataloader: DataLoader) -> float:
    model.eval_mode()

    with torch.no_grad():  # Disable gradient computation
        all_correct: int = 0
        for inputs, labels in dataloader:
            # Move inputs and labels to the specified device
            inputs, labels = inputs.to(device), labels.to(device)
            outputs: torch.Tensor = model(inputs)
            comparison: torch.Tensor = torch.argmax(outputs, axis=1) == torch.argmax(
                labels, axis=1
            )
            all_correct += sum(comparison)
        accuracy: float = all_correct / len(dataloader.dataset)
    model.train_mode()
    return accuracy
test_data = test_data.to(device)
model.eval_mode()
model(test_data[0:1])

d = test_data[0:1]

print(model.second_layer(model.first_layer(d)))
#
# # model(test_data[0,...])
print(get_signed_accuracy(model, val_dataloader))


tensor([[ 1.,  1.,  1., -1., -1.,  1., -1.,  1., -1.,  1.,  1., -1., -1., -1.,
          1., -1., -1., -1., -1., -1.,  1., -1.,  1.,  1., -1.,  1., -1.,  1.,
         -1., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1., -1.,  1.,  1.,  1.,
          1.,  1., -1., -1.,  1., -1., -1.,  1., -1., -1.,  1.,  1.]],
       device='cuda:0', grad_fn=<SignBackward0>)
tensor(0.9447, device='cuda:0')


In [6]:
save_model(model, "convnet")